# Упражнение: Патчи PatchTST и каузальное внимание по ним

## Задание

В `input.csv` — окно истории из 48 точек (`t, y`). В `weights.json` — веса крошечной «головы внимания»:
`W_emb` (8 × 4), `b_emb` (4), `W_q`, `W_k` (4 × 4), `W_v` (4 × 2).

1. **Нормализация окна** (RevIN без обучаемых параметров): z = (y − mean) / std по всем 48 точкам, std с `ddof=0`.
2. **Патчи:** длина P = 8, шаг S = 4 (патчи перекрываются). Патч i покрывает моменты
   4i … 4i + 7; число патчей N = (48 − 8) / 4 + 1 = 11.
3. **Эмбеддинг:** E = patches · W_emb + b_emb (11 × 4) — каждый патч становится токеном.
4. **Самовнимание:** Q = E·W_q, K = E·W_k, V = E·W_v; scores = Q·Kᵀ / √4.
5. **Каузальная маска:** токен i видит только токены j ≤ i (их патчи заканчиваются не позже его патча);
   для j > i scores = −∞. Веса A = softmax по строке, контекст ctx = A·V (11 × 2).

## Подсказки

- Первая строка внимания тривиальна: токену 0 виден только он сам, его вес равен 1.
- Маска — верхний треугольник без диагонали: `np.triu(np.ones((N, N), bool), k=1)`.
  Удобно заменить там scores на `-np.inf` и делать softmax с вычитанием максимума строки.
- Патчи: `np.stack([z[s:s + 8] for s in range(0, 41, 4)])`. Для каждой строки i выведите номер ключа с
  максимальным весом (`argmax`), этот вес, вес «на себя» A[i, i] и две координаты контекста.

## Ожидаемый формат выходного файла

CSV с колонками `patch, t_start, t_end, top_key, top_weight, self_weight, ctx1, ctx2` — 11 строк
(patch = 0..10 по порядку; `t_start, t_end` — первый и последний момент патча; `top_key` — номер патча).
Вещественные значения округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/transformers_time_series
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.